# pgvector Face Embedding Sync - Test Notebook

This notebook tests the pgvector integration for face recognition embeddings.

## Setup Instructions

1. Start PostgreSQL: `docker compose up postgres -d`
2. Run this notebook to test sync operations
3. Verify embeddings are stored in pgvector

## What This Tests

- Database connection
- User sync (add/update/delete)
- Embedding calculation
- Similarity search
- Integration with backend notifications

## 📦 Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install psycopg2-binary sqlalchemy pgvector python-dotenv requests --quiet

print('✅ Dependencies installed!')

✅ Dependencies installed!


## 🚀 Setup

In [ ]:
import os
import sys
import requests
import json
from datetime import datetime
from dotenv import load_dotenv

# Add project root to path
sys.path.insert(0, '..')

load_dotenv()

# Configuration
FR_API_URL = os.getenv('FACE_RECOGNITION_API_URL', 'http://localhost:8000')
BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')
CLIENT_SLUG = 'dev'  # Change to your client

print('🎉 Setup complete!')
print(f'✅ FR API: {FR_API_URL}')
print(f'✅ Backend API: {BACKEND_URL}')
print(f'✅ Client: {CLIENT_SLUG}')

🎉 Setup complete!
✅ FR API: http://localhost:8000
✅ Backend API: http://localhost:7091
✅ Client: dev


## Test 1: Database Connection

In [3]:
from src.face_recognition.storage.db_config import DatabaseConfig

print('🔍 Testing database connection...')

db_config = DatabaseConfig()
success = db_config.test_connection()

if success:
    print('✅ Database connection successful!')
    print(f'   Host: {db_config.host}:{db_config.port}')
    print(f'   Database: {db_config.database}')
else:
    print('❌ Database connection failed!')
    print('   Make sure PostgreSQL is running: docker compose up postgres -d')

ModuleNotFoundError: No module named 'loguru'

## Test 2: Initialize Schema

In [4]:
print(f'🔧 Initializing schema for client: {CLIENT_SLUG}')

db_config.init_schema(CLIENT_SLUG)

print(f'✅ Schema org_{CLIENT_SLUG} initialized!')
print('   Tables: face_embeddings')
print('   Indexes: user_id, embedding (ivfflat)')

🔧 Initializing schema for client: dev


NameError: name 'db_config' is not defined

## Test 3: pgvector Store Operations

In [ ]:
from src.face_recognition.storage.pgvector_store import PgVectorStore
import numpy as np

print('🧪 Testing pgvector store operations...')

store = PgVectorStore(CLIENT_SLUG)

# Get current count
current_count = store.get_embedding_count()
print(f'Current embeddings in database: {current_count}')

### Test 3a: Add Test Embedding

In [ ]:
# Create a test embedding (512-dim)
test_embedding = np.random.rand(512).astype(np.float32)
test_embedding = test_embedding / np.linalg.norm(test_embedding)  # Normalize

# Add to database
embedding_id = store.add_embedding(
    user_id='test_user_1',
    user_name='Test User',
    image_url='test://image.jpg',
    embedding=test_embedding,
    external_id='EMP001',
    metadata={'test': True}
)

print(f'✅ Added test embedding with ID: {embedding_id}')

# Verify count increased
new_count = store.get_embedding_count()
print(f'Total embeddings: {new_count} (was {current_count})')

### Test 3b: Search Similar Embeddings

In [ ]:
# Search for similar embeddings
print('🔍 Searching for similar faces...')

matches = store.search_similar(
    query_embedding=test_embedding,
    limit=5,
    threshold=0.0  # Low threshold to see all results
)

print(f'Found {len(matches)} matches:')
for match in matches:
    print(f"  - {match['user_name']} (ID: {match['user_id']}): {match['similarity']:.4f}")

### Test 3c: Delete Embedding

In [ ]:
# Delete test embedding
deleted = store.delete_all_for_user('test_user_1')
print(f'✅ Deleted {deleted} embedding(s) for test_user_1')

final_count = store.get_embedding_count()
print(f'Final count: {final_count}')

## Test 4: Embedding Sync Service

In [ ]:
print('🔄 Testing embedding sync service...')

# Test with mock user data
mock_user_data = {
    'id': '999',
    'full_name': 'Test Sync User',
    'external_id': 'EMP999',
    'image_urls': [
        {'original': 'https://example.com/test.jpg'}  # This will fail but that's OK for testing
    ]
}

from src.face_recognition.services.embedding_sync import EmbeddingSyncService

sync_service = EmbeddingSyncService(CLIENT_SLUG, gpu_id=0)

print('✅ Sync service initialized')
print('   Note: Full sync requires GPU and real images')

## Test 5: API Endpoints

### Test 5a: Health Check

In [ ]:
print('🏥 Testing FR API health endpoint...')

try:
    response = requests.get(f'{FR_API_URL}/api/v1/health', timeout=5)

    if response.status_code == 200:
        health = response.json()
        print('✅ FR API is healthy!')
        print(f'   Status: {health.get("status")}')
        print(f'   Active clients: {health.get("active_clients")}')
    else:
        print(f'⚠️  FR API returned status {response.status_code}')

except requests.exceptions.ConnectionError:
    print('❌ FR API is not running')
    print('   Start it with: python src/web/api.py')

### Test 5b: Sync Add User (requires FR API running)

In [ ]:
# This test requires the FR API to be running
# Skip if API is not available

print('📤 Testing sync/add_user endpoint...')

test_payload = {
    'client_slug': CLIENT_SLUG,
    'user_data': {
        'id': '888',
        'full_name': 'API Test User',
        'external_id': 'EMP888',
        'image_urls': []
    }
}

try:
    response = requests.post(
        f'{FR_API_URL}/api/v1/sync/add_user',
        json=test_payload,
        timeout=30
    )

    if response.status_code == 200:
        result = response.json()
        print('✅ Sync add_user successful!')
        print(json.dumps(result, indent=2))
    else:
        print(f'❌ Failed: {response.status_code}')
        print(response.text)

except requests.exceptions.ConnectionError:
    print('⚠️  Skipping - FR API not running')

## Test 6: Migration Script

In [ ]:
print('📋 Migration script location:')
print('   scripts/migrate_pkl_to_pgvector.py')
print()
print('Usage:')
print('   python scripts/migrate_pkl_to_pgvector.py --client dev --dry-run')
print('   python scripts/migrate_pkl_to_pgvector.py --all')

## Summary

✅ **Tests Complete!**

### What We Tested:
1. Database connection
2. Schema initialization
3. pgvector CRUD operations
4. Similarity search
5. Embedding sync service
6. API endpoints

### Next Steps:
1. Set `USE_PGVECTOR=true` in .env
2. Place GCS credentials in `./keys/gcs-service-account.json`
3. Run migration: `python scripts/migrate_pkl_to_pgvector.py --all`
4. Test with real backend integration using `test_fr_api.ipynb`

### Backend Integration:
Backend should call these endpoints:
- `POST /api/v1/sync/add_user` - When user created
- `POST /api/v1/sync/update_user` - When user updated
- `POST /api/v1/sync/delete_user` - When user deleted
- `POST /api/v1/sync/rebuild` - Rebuild all embeddings